# **Build the fist MIL pipeline : 1D CNN + Classifier**

In [33]:
from pathlib import Path
CHECKPOINT_DIR = Path("checkpoints_mil")


In [34]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Dossier des checkpoints :", CHECKPOINT_DIR.resolve())

Dossier des checkpoints : C:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\Base_line\checkpoints_mil


In [35]:
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    precision_score
)
import torch

In [36]:
import os

# Enable precise CUDA kernel error reporting
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

# Import PyTorch AFTER setting the environment variables
import torch

In [37]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; refusing to train on CPU.")
print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [38]:
from Data_pre_processing.data_cleaning import data_cleaning, split_train_test
import torch
from torch.utils.data import Dataset
from Data_pre_processing.Signal_treatement import split_signals_and_build

## **Building the dataset**

In [39]:
class PCGDataset(Dataset):
    def __init__(self, bags, labels):
        self.bags = (bags)
        self.labels = torch.as_tensor(labels, dtype=torch.float32)
        assert len(bags) == len(labels)

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):

        bag = torch.as_tensor(
            self.bags[idx], dtype=torch.float32
        )
        label = self.labels[idx]

        return bag, label

## **Building the pipeline**

In [40]:
class Mil_classifier(nn.Module) :
    def __init__(self):
        # signal = [[],[],[]] = 1
        super(Mil_classifier, self).__init__()
        self.conv1 = nn.Conv1d(1, 20, 100, stride=40)
        self.bn1 = nn.BatchNorm1d(20)
        self.pool1 = nn.MaxPool1d(15, 6)

        self.conv2 = nn.Conv1d(20, 40, 5)
        self.bn2 = nn.BatchNorm1d(40)
        self.pool2 = nn.MaxPool1d(5, 2)

        self.conv3 = nn.Conv1d(40, 20, 3, padding=1)
        self.bn3 = nn.BatchNorm1d(20)
        self.pool3 = nn.MaxPool1d(3, 2)

        self.gap = nn.AdaptiveAvgPool1d(1)

        self.fc1 = nn.Linear(20, 10)
        self.fc2 = nn.Linear(10,1)

    def forward(self, x : torch.tensor):
        # Input: [B, N, L]
        B, N, L = x.shape

        # Treat every segment as an individual signal.
        # [B, N, L] -> [B*N, 1, L]
        x = x.reshape(B * N, 1, L)

        x = self.bn1(self.conv1(x))
        x = self.pool1(F.relu(x))

        x = self.bn2(self.conv2(x))
        x = self.pool2(F.relu(x))

        x = self.bn3(self.conv3(x))
        x = self.pool3(F.relu(x))

        x = self.gap(x).squeeze(-1)
        x = F.relu(self.fc1(x))

        
        x = self.fc2(x)
        x = x.reshape(B, N)

        return x

        # x = x.max(dim=0).values
        # return x

## **Preparing the data**

In [41]:
counts, data = data_cleaning()

In [42]:
data = data.reset_index(drop=True)

In [43]:
data_training, data_testing = split_train_test(data=data)

Normal: 522
Abnormal: 426


#### **Training set**

In [44]:
data_training_splited = split_signals_and_build(data=data_training, segment_duration=5)

In [45]:
signals_dict = data_training_splited.groupby('file')['signal'].apply(lambda x: np.array(x.tolist())).to_dict()
dataset_training_x = (list(signals_dict.values()))
dataset_training_y = np.int32(data_training.iloc[:,1].values)
dataset_training_y = (dataset_training_y > 0).astype(np.int64)

#### **Test set**

In [46]:
data_test_splited = split_signals_and_build(data=data_testing, segment_duration=5)
signals_dict_test = data_test_splited.groupby('file')['signal'].apply(lambda x: np.array(x.tolist())).to_dict()
dataset_testing_x = (list(signals_dict_test.values()))
dataset_testing_y = np.int32(data_testing.iloc[:,1].values)
dataset_testing_y = (dataset_testing_y > 0).astype(np.int64)

In [47]:
i=0
for d in dataset_testing_y:
    if d == 1:
        i+=1

print(i)

146


### Applying the normalization

In [48]:
# import numpy as np

# signals_normalized = []

# for signal in dataset_training_x:
#     min_val = signal.min()
#     max_val = signal.max()

#     if max_val > min_val:
#         normalized_signal = (signal - min_val) / (max_val - min_val)
#     else:
#         normalized_signal = np.zeros_like(signal)

#     signals_normalized.append(normalized_signal)

In [49]:
# signals_normalized[0]

In [50]:
# dataset_training_x = signals_normalized

In [51]:
dataset = PCGDataset(dataset_training_x, dataset_training_y)

In [52]:
dataset_test = PCGDataset(dataset_testing_x, dataset_testing_y)

In [53]:
test_loader = DataLoader(
      dataset_test,
      batch_size=1,
      shuffle=True
  )

## **Training**

For the training, since we got an umbalanced data, we are going the stratified cross validation, and that gonna help the model to genelize on all the existing classes in the dataset.

### Stratified cross validation

In [54]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

- 1 for abnormal, 0 for normal.
- In this case we are going to priviluaze our folds to see the data distribution

In [55]:
folds = []

for train_index, test_index in skf.split(
    np.zeros(len(dataset_training_y)),
    dataset_training_y,
):
    folds.append((train_index, test_index))
  
for i, (train_index, test_index) in enumerate(folds):
  print(f"Fold {i} : ")
  print(f"Train labels")
  numbers_of_zero = 0
  numbers_of_one = 0
  for index in train_index :
    if dataset_training_y[index] == 1 :
      numbers_of_one+=1
    else : numbers_of_zero+=1
  print(f"numbers of abnormal : {numbers_of_one}, numbers of normal : {numbers_of_zero}")
  numbers_of_zero = 0
  numbers_of_one = 0
  for index in test_index :
    if dataset_training_y[index] == 1 :
      numbers_of_one+=1
    else : numbers_of_zero+=1
  print(f"Test labels")
  print(f"numbers of abnormal : {numbers_of_one}, numbers of normal : {numbers_of_zero}")

Fold 0 : 
Train labels
numbers of abnormal : 340, numbers of normal : 418
Test labels
numbers of abnormal : 86, numbers of normal : 104
Fold 1 : 
Train labels
numbers of abnormal : 341, numbers of normal : 417
Test labels
numbers of abnormal : 85, numbers of normal : 105
Fold 2 : 
Train labels
numbers of abnormal : 341, numbers of normal : 417
Test labels
numbers of abnormal : 85, numbers of normal : 105
Fold 3 : 
Train labels
numbers of abnormal : 341, numbers of normal : 418
Test labels
numbers of abnormal : 85, numbers of normal : 104
Fold 4 : 
Train labels
numbers of abnormal : 341, numbers of normal : 418
Test labels
numbers of abnormal : 85, numbers of normal : 104


### Defining the Hyper-parameters of the model

In [77]:
NUM_EPOCHS = 45
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-3
fold_results = []

### **Training phase**

#### Function to build data loaders

In [57]:
def build_data_loaders(dataset, train_index, test_index):
  training_dataset = Subset(dataset, train_index.tolist())
  valid_dataset = Subset(dataset, test_index.tolist())
  #Building the train loader
  train_loader = DataLoader(
      training_dataset,
      batch_size=1,
      # shuffle=True
  )
  #Building the validation loader
  valid_loader = DataLoader(
      valid_dataset,
      batch_size=1,
      shuffle=False
  )

  return train_loader, valid_loader

#### Function to train the model

In [58]:
def train_mil(optimizer, criterion, start_epoch, num_epochs, train_loader, model, checkpoint_path):
  loss_res = []
  for epoch in range(start_epoch, num_epochs):
    logits=[]
    model.train()

    total_loss = 0.0
    num_bags = 0

    for bag, label in train_loader:
      bag = bag.squeeze(0).unsqueeze(1).to(device)
      label = label.to(device).reshape(())

      optimizer.zero_grad()
      # One logit for this entire recording.
      output = model(bag)
      logit = output.mean(dim=0)

      
      loss = criterion(
          logit,
          label.float().reshape_as(logit)
      )
      # print("Loss:", loss.item())
      loss.backward()
      
      optimizer.step()
      total_loss += loss.item()
      num_bags += 1

    mean_loss = total_loss / num_bags
    loss_res.append(mean_loss)
    print(
            f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} "
            f"| Train loss: {mean_loss:.4f}"
        )
    checkpoint = {
            "fold": fold,
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": loss,
            "num_epochs_target": NUM_EPOCHS
        }
    
    torch.save(checkpoint, checkpoint_path)
    print("checkpoint saved !")

In [59]:
def eval_mil(valid_loader, model) :
  model.eval()

  all_labels = []
  all_probs = []

  with torch.no_grad():
    for bag, label in valid_loader :
      bag = bag.squeeze(0).unsqueeze(1).to(device)
      output = model(bag)
      logit = output.mean(dim=0)
      proba = torch.sigmoid(logit).item()
      all_probs.append(proba)
      all_labels.append(int(label.item()))
      

    y_true = np.asarray(all_labels, dtype=np.int64)
    y_prob = np.asarray(all_probs)
    y_pred = (y_prob >= 0.5).astype(np.int64)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()
    print("Confusion Matrix :")
    print(f"True negative {tn}")
    print(f"True positive {tp}")
    print(f"False negative {fn}")
    print(f"False positive {fp}")

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred), #Measure the global score of the model, we can't just rely on this metric
        "precision" : precision_score(y_true=y_true, y_pred=y_pred, pos_label=1), #Measure the level of confidence of the model by predicting the abnormal class,
        "precision - abnormal" : tp / (tp+fp), #Measure the level of confidence of the model by predicting the abnormal class,
        "sensitivity - recall abnormal": tp / (tp+fn),
        "precision - noraml" : tn / (tn+fn), #Measure the level of confidence of the model by predicting the abnormal class,
        "sensitivity - recall normal": tn / (tn+fp),
        }

    fold_results.append(metrics)
    print(f"Validation bags: {len(y_true)}")

    for name, value in metrics.items():
        print(f"{name}: {value:.4f}")

#### Training and validation

In [88]:
folds = []
pos_weight = torch.tensor([1.22], device=device)

for train_index, test_index in skf.split(
    np.zeros(len(dataset_training_y)),
    dataset_training_y,
):
    folds.append((train_index, test_index))

for fold, (train_idx, valid_idx) in enumerate(folds) : 
    print(f"training on the fold {fold}")
    print(f"*"*20, "INFORMATION ABOUT THE TRAINING", "*"*20)

    numbers_of_zero = 0
    numbers_of_one = 0
    for index in train_index :
      if dataset_training_y[index] == 1 :
        numbers_of_one+=1
      else : numbers_of_zero+=1
    print(f"      numbers of abnormal : {numbers_of_one}, numbers of normal : {numbers_of_zero}")

    torch.manual_seed(100 + fold)

    model = Mil_classifier().to(device)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    checkpoint_path = CHECKPOINT_DIR / f"fold_{fold}.pt"

    start_epoch = 0

    if checkpoint_path.exists():
        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True
        )

        if checkpoint["fold"] != fold:
            raise ValueError("Le checkpoint ne correspond pas au fold.")

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        start_epoch = checkpoint["epoch"]

        print(
            f"Reprise du fold {fold} : "
            f"dernière époque terminée = {start_epoch}"
        )
    else:
        print(f"New training for the fold {fold}")

    train_loader, valid_loader = build_data_loaders(dataset=dataset, train_index=train_index, test_index=test_index)

    train_mil(optimizer=optimizer, criterion=criterion, start_epoch=start_epoch, num_epochs=NUM_EPOCHS, train_loader=train_loader, model=model, checkpoint_path = checkpoint_path)

    eval_mil(valid_loader = valid_loader, model=model)
    

training on the fold 0
******************** INFORMATION ABOUT THE TRAINING ********************
      numbers of abnormal : 341, numbers of normal : 418
Reprise du fold 0 : dernière époque terminée = 60
Confusion Matrix :
True negative 41
True positive 23
False negative 62
False positive 63
Validation bags: 189
accuracy: 0.3386
precision: 0.2674
precision - abnormal: 0.2674
sensitivity - recall abnormal: 0.2706
precision - noraml: 0.3981
sensitivity - recall normal: 0.3942
training on the fold 1
******************** INFORMATION ABOUT THE TRAINING ********************
      numbers of abnormal : 341, numbers of normal : 418
Reprise du fold 1 : dernière époque terminée = 45
Confusion Matrix :
True negative 78
True positive 55
False negative 30
False positive 26
Validation bags: 189
accuracy: 0.7037
precision: 0.6790
precision - abnormal: 0.6790
sensitivity - recall abnormal: 0.6471
precision - noraml: 0.7222
sensitivity - recall normal: 0.7500
training on the fold 2
******************** 

KeyboardInterrupt: 

In [84]:
model_test = Mil_classifier().to(device)

checkpoint_path_test = CHECKPOINT_DIR / "fold_2.pt"

checkpoint_test = torch.load(
    checkpoint_path_test,
    map_location=device,
    weights_only=True
)

# Load trained weights into the model
model_test.load_state_dict(checkpoint_test["model_state_dict"])

<All keys matched successfully>

In [85]:
def evaluate_test_loss(model, test_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for signals, labels in test_loader:
            signals = signals.squeeze(0).unsqueeze(1).to(device)
            labels = labels.float().to(device)

            # Forward pass: segment-level logits
            output = model(signals)

            # Aggregate segment logits into one recording-level logit
            logit = output.mean(dim=0)


            proba = torch.sigmoid(logit).item()
            all_probs.append(proba)
            all_labels.append(int(labels.item()))

    y_true = np.asarray(all_labels, dtype=np.int64)
    y_prob = np.asarray(all_probs)
    y_pred = (y_prob >= 0.5).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=[0, 1]
    ).ravel()
    print("Confusion Matrix :")
    print(f"True negative {tn}")
    print(f"True positive {tp}")
    print(f"False negative {fn}")
    print(f"False positive {fp}")

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred), #Measure the global score of the model, we can't just rely on this metric
        "precision" : precision_score(y_true=y_true, y_pred=y_pred, pos_label=1), #Measure the level of confidence of the model by predicting the abnormal class,
        "precision - abnormal" : tp / (tp+fp), #Measure the level of confidence of the model by predicting the abnormal class,
        "sensitivity - recall abnormal": tp / (tp+fn),
        "precision - noraml" : tn / (tn+fn), #Measure the level of confidence of the model by predicting the abnormal class,
        "sensitivity - recall normal": tn / (tn+fp),
        }
    print(metrics)
    

In [86]:
pos_weight = torch.tensor([1.23], device=device)

In [87]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
evaluate_test_loss(model=model_test, test_loader=test_loader, criterion=criterion, device=device)

Confusion Matrix :
True negative 1179
True positive 105
False negative 41
False positive 601
{'accuracy': 0.6666666666666666, 'precision': 0.14872521246458922, 'precision - abnormal': np.float64(0.14872521246458922), 'sensitivity - recall abnormal': np.float64(0.7191780821917808), 'precision - noraml': np.float64(0.9663934426229508), 'sensitivity - recall normal': np.float64(0.6623595505617977)}
